# AHR999 BTC Hoarding Index Quickstart

This notebook loads the daily-updated Kaggle mirror of `ahr999-dataset`, prints the latest reading, and renders a compact overview chart inspired by the canonical dashboard at https://ahr999.aix4u.com/.

The data is for research, education, and observability only. It is not financial advice.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


kaggle_input_paths = sorted(Path('/kaggle/input').rglob('ahr999.csv')) if Path('/kaggle/input').exists() else []
candidate_paths = [
    Path('/kaggle/input/ahr999-btc-hoarding-index-dataset/ahr999.csv'),
    *kaggle_input_paths,
    Path('datasets/ahr999.csv'),
    Path('../../../datasets/ahr999.csv'),
    Path('../../datasets/ahr999.csv'),
]
csv_path = next((path for path in candidate_paths if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError('Could not find ahr999.csv in Kaggle input or local repo paths.')

df = pd.read_csv(csv_path, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df.tail()

In [ ]:
def regime_for(value):
    if pd.isna(value):
        return 'n/a', '#a1a1aa', '-'
    if value < 0.45:
        return 'Bargain zone', '#f87171', '< 0.45'
    if value < 1.20:
        return 'DCA zone', '#34d399', '0.45 - 1.20'
    if value < 3.00:
        return 'Caution zone', '#f59e0b', '1.20 - 3.00'
    return 'Bubble zone', '#fb7185', '> 3.00'


valid = df.dropna(subset=['ahr999']).copy()
latest = valid.iloc[-1]
week_ago = valid[valid['date'] <= latest['date'] - pd.Timedelta(days=7)].tail(1)
delta_7d = np.nan if week_ago.empty else latest['ahr999'] - week_ago.iloc[0]['ahr999']
regime_label, regime_color, regime_range = regime_for(latest['ahr999'])

summary = pd.DataFrame(
    [
        ['date', latest['date'].strftime('%Y-%m-%d')],
        ['ahr999', f"{latest['ahr999']:.4f}"],
        ['regime', regime_label],
        ['btc_close_usd', f"${latest['close']:,.2f}"],
        ['ma200_usd', f"${latest['ma200']:,.2f}"],
        ['quantile_5y', f"{latest['quantile5y'] * 100:.1f}%"],
        ['delta_7d_ahr999', 'n/a' if pd.isna(delta_7d) else f"{delta_7d:+.4f}"],
        ['rows', f"{len(df):,}"],
    ],
    columns=['metric', 'value'],
)
summary

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter


COLOR_BG = '#050506'
COLOR_PANEL = '#09090b'
COLOR_GRID = '#27272a'
COLOR_PRICE = '#f59e0b'
COLOR_MA200 = '#94a3b8'
COLOR_AHR = '#60a5fa'
COLOR_FLOOR = '#f87171'
COLOR_DCA = '#34d399'
COLOR_FG = '#fafafa'
COLOR_MUTED = '#a1a1aa'

plot_df = valid.tail(365 * 2).copy()

plt.rcParams.update({
    'figure.facecolor': COLOR_BG,
    'axes.facecolor': COLOR_PANEL,
    'axes.edgecolor': COLOR_GRID,
    'axes.labelcolor': COLOR_MUTED,
    'xtick.color': COLOR_MUTED,
    'ytick.color': COLOR_MUTED,
    'text.color': COLOR_FG,
    'font.family': 'DejaVu Sans',
})

fig = plt.figure(figsize=(15, 9), dpi=150)
gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.55, 4.6], hspace=0.22)

headline = fig.add_subplot(gs[0])
headline.set_facecolor(COLOR_BG)
headline.axis('off')
headline.text(0.00, 0.86, f"AHR999 - latest reading UTC {latest['date'].strftime('%Y-%m-%d')}", fontsize=10, color=COLOR_MUTED, weight='bold', transform=headline.transAxes)
headline.text(0.00, 0.18, f"{latest['ahr999']:.4f}", fontsize=60, color=COLOR_FG, transform=headline.transAxes)
headline.text(0.28, 0.42, '•', fontsize=22, color=regime_color, transform=headline.transAxes)
headline.text(0.305, 0.44, regime_label, fontsize=16, color=regime_color, weight='bold', transform=headline.transAxes)
headline.text(0.43, 0.44, regime_range, fontsize=14, color=COLOR_MUTED, transform=headline.transAxes)
headline.text(0.00, 0.04, 'Self-computed daily from Binance BTCUSDT closes. Canonical source: https://ahr999.aix4u.com/', fontsize=11, color=COLOR_MUTED, transform=headline.transAxes)

metrics = [
    ('BTC CLOSE', f"${latest['close']:,.2f}"),
    ('MA200', f"${latest['ma200']:,.2f}"),
    ('5Y QUANTILE', f"{latest['quantile5y'] * 100:.1f}%"),
    ('7D AHR', 'n/a' if pd.isna(delta_7d) else f"{delta_7d:+.4f}"),
]
for i, (label, value) in enumerate(metrics):
    x = 0.58 + i * 0.105
    headline.text(x, 0.70, label, fontsize=8, color=COLOR_MUTED, weight='bold', transform=headline.transAxes)
    headline.text(x, 0.44, value, fontsize=12, color=COLOR_FG if i != 3 else COLOR_DCA, transform=headline.transAxes)

ax = fig.add_subplot(gs[1])
ax2 = ax.twinx()

ax.plot(plot_df['date'], plot_df['ma200'], color=COLOR_MA200, linewidth=1.4, alpha=0.72, label='MA200 - left')
ax.plot(plot_df['date'], plot_df['close'], color=COLOR_PRICE, linewidth=1.7, label='BTC price - left')
ax2.plot(plot_df['date'], plot_df['ahr999'], color=COLOR_AHR, linewidth=1.6, label='AHR999 - right')

ax.set_yscale('log')
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"${value/1000:.0f}k" if value >= 1000 else f"${value:.0f}"))
ax.yaxis.set_minor_formatter(NullFormatter())
ax2.set_ylim(0, max(1.8, float(plot_df['ahr999'].max()) * 1.08))
ax2.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.1f}"))

ax2.axhline(0.45, color=COLOR_FLOOR, linewidth=1.0, linestyle='--', alpha=0.85, label='_nolegend_')
ax2.axhline(1.20, color=COLOR_DCA, linewidth=1.0, linestyle='--', alpha=0.85, label='_nolegend_')
threshold_label_x = plot_df['date'].iloc[int(len(plot_df) * 0.94)]
ax2.text(threshold_label_x, 0.45, 'Bargain - 0.45', color=COLOR_FLOOR, fontsize=9, va='bottom')
ax2.text(threshold_label_x, 1.20, 'DCA - 1.20', color=COLOR_DCA, fontsize=9, va='bottom')

ax.grid(True, which='major', axis='both', color=COLOR_GRID, linestyle='--', linewidth=0.65, alpha=0.55)
ax.set_title('BTCUSDT daily close, MA200, and AHR999 threshold zones', loc='left', fontsize=12, color=COLOR_MUTED, pad=18)
ax.set_xlabel('UTC close date')
ax.set_ylabel('BTC price / MA200, log scale')
ax2.set_ylabel('AHR999')

lines = [ax.get_lines()[0], ax.get_lines()[1], ax2.get_lines()[0]]
labels = ['MA200 - left', 'BTC price - left', 'AHR999 - right']
legend = ax.legend(lines, labels, loc='upper left', ncols=3, frameon=True, facecolor=COLOR_PANEL, edgecolor=COLOR_GRID)
for text in legend.get_texts():
    text.set_color(COLOR_MUTED)

for spine in list(ax.spines.values()) + list(ax2.spines.values()):
    spine.set_color(COLOR_GRID)

fig.tight_layout()
fig.savefig('ahr999-overview.png', facecolor=fig.get_facecolor(), bbox_inches='tight')
plt.show()

## Fetch the latest row outside Kaggle

```bash
curl -s https://ahr999.aix4u.com/datasets/ahr999.json | jq '.[-1]'
```

Canonical repo: https://github.com/RuochenLyu/ahr999-dataset  
Canonical dashboard: https://ahr999.aix4u.com/  
Archival snapshot DOI: https://doi.org/10.5281/zenodo.20412604